# Train and Validate
* Train a model across the specified training sites and sampling approach
* Validate the model at the excluded site

## Todo
* Pull out probabilities using model.predict_proba(observations_to_predict)

In [ ]:
import pathlib
import numpy
import dask.distributed
import pandas
import joblib

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

# Values to edit

In [ ]:
sample_method = "sampling_2"
method_2_threshold = .95 # .98 .99 .97 .95

In [ ]:
# View the amount of training data for the selected sampling apporach
samples_per_site = pandas.read_csv(utils.get_samples_summary_file_path(sample_method, method_2_threshold))
samples_per_site

In [ ]:
all_training_sites =  ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

In [ ]:
uav_classes_to_ignore = ['Shadow', 'Glare']

In [ ]:
# Mappings of Satellite classes to consider from UAV classes
satellite_4_classes = {'Seagrass': 1, 'Mixed': 4, 'Unvegetated': 5, 'Water': 6,}
satellite_4_classes_from_uav_classes = {'Seagrass': ['Seagrass', 'Seagrass submerged'],
'Mixed': ['Cystophora', 'Hormosira', 'Submerged vegetation','Brown algae mixed', 'Microphytobenthos', 'Green algae mixed', 
          'Filamentous brown algae', 'Red algae mixed', 'Saltmarsh', 
          'Gracilaria', 'Gracilaria submerged', 'Ulva', 'Ulva mats'],
  'Unvegetated': [ 'Terrestrial', 'Unvegetated', 'Rock'],                            
  'Water': ['Water'],
}

In [ ]:
# Mappings of Satellite classes to consider from UAV classes
satellite_5_classes = {'Seagrass': 1, 'Gracilaria': 2, 'Mixed': 4, 'Unvegetated': 5, 'Water': 6,}
satellite_5_classes_from_uav_classes = {'Seagrass': ['Seagrass', 'Seagrass submerged'],
                              'Gracilaria': ['Gracilaria', 'Gracilaria submerged'],
'Mixed': ['Cystophora', 'Hormosira', 'Submerged vegetation','Brown algae mixed', 'Microphytobenthos', 'Green algae mixed', 
          'Filamentous brown algae', 'Red algae mixed', 'Saltmarsh',
          'Ulva', 'Ulva mats'],
  'Unvegetated': [ 'Terrestrial', 'Unvegetated', 'Rock'],                            
  'Water': ['Water'],
}

In [ ]:
# Mappings of Satellite classes to consider from UAV classes
satellite_6_classes = {'Seagrass': 1, 'Gracilaria': 2, 'Ulva': 3, 'Mixed': 4, 'Unvegetated': 5, 'Water': 6,}
satellite_6_classes_from_uav_classes = {'Seagrass': ['Seagrass', 'Seagrass submerged'],
                              'Gracilaria': ['Gracilaria', 'Gracilaria submerged'],
                              'Ulva': ['Ulva', 'Ulva mats'],
'Mixed': ['Cystophora', 'Hormosira', 'Submerged vegetation','Brown algae mixed', 'Microphytobenthos', 'Green algae mixed', 
           'Filamentous brown algae', 'Red algae mixed', 'Saltmarsh'],
  'Unvegetated': [ 'Terrestrial', 'Unvegetated', 'Rock'],                            
  'Water': ['Water'],
}

In [ ]:
training_sites_akaroa = ["Duvauchelle", "Robinsons", "Childrens", "Takamatua"] 
training_sites_SI = ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai"] 

### Make sure you change the `model_file` name when experimenting

In [ ]:
satellite_classes=satellite_5_classes
satellite_from_uav_classes=satellite_5_classes_from_uav_classes

In [ ]:
test_site = "Ihutai"
training_sites = training_sites_SI
model_file = utils.get_model_file(
    f"trained_on_akaroa_sites_thresh_{int(method_2_threshold*100)}_tested_on_{test_site}_5_class")

# Ensure not training on test site
if test_site in training_sites:
    training_sites.remove(test_site)


# Cells to run
* Train and save model
* Review model
  * satellite bands of each UAV class
  * satellite bands of each satellite class
  * importance of satellite bands in trained model
* Validation
  * Predict excluded site
  * Plot confusion matrix comparing prediction to UAV classifications

In [ ]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()
uav_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
sample_folder = utils.get_samples_path(sample_method=sample_method, method_2_threshold=method_2_threshold)

test_satellite_file = data_path / "training" / "satellite_images" / f"{test_site}_sentinel-2.nc"
test_polygon_file = polygon_file = utils.get_site_polygon_path(test_site)
test_uav_file = data_path / "classified_orthos" / f"{test_site}_classified.tif"

test_prediction_file = data_path / "validation" / "predictions" / f"{test_site}_prediction_{model_file.stem}.nc"

### Train and save model

In [ ]:
if not model_file.exists():
    # train and save the model
    model, training_dataframe = training.train_classifier(
        training_sites=training_sites,
        samples_path=sample_folder,
        uav_labels_file=uav_labels_file,
        uav_classes_to_ignore=uav_classes_to_ignore,
        satellite_classes=satellite_classes,
        satellite_from_uav_classes=satellite_from_uav_classes)
    
    # Save model
    joblib.dump(model, model_file); # , compress=3);
    
    # Save a record of the classes considered in training
    satellite_classes_dataframe = pandas.DataFrame.from_dict(satellite_classes, orient='index', columns=['satellite_class_id'])
    satellite_classes_dataframe['uav_class_ids'] = satellite_classes_dataframe.index.map(lambda key: f"{satellite_from_uav_classes[key]}")
    satellite_classes_dataframe.to_csv(model_file.with_name(f"{model_file.stem}_class_mappings.csv"), index=True)

    # Save a summary of the training data
    training_data_summary=pandas.DataFrame(training_dataframe['satellite_class_id'].value_counts())
    training_data_summary["satellite_class_name"] = training_data_summary.index.map(lambda index: next((key for key, value in satellite_classes.items() if value == int(index)), None) )
    training_data_summary = training_data_summary[["satellite_class_name", "count"]] # Change column order
    training_data_summary.to_csv(model_file.with_name(f"{model_file.stem}_training_data_summary.csv"))
    training_data_summary
    

else:
    print(f"Model '{model_file.name}' already exists. Delete if you want to recreate it.")
    training_data_summary = pandas.read_csv(model_file.with_name(f"{model_file.stem}_training_data_summary.csv"))
    training_dataframe = None
    

print(f"\nSatellite training classes present: {list(training_data_summary['satellite_class_name'])}")

### Review model

In [ ]:
training.plot_training_data_class_distribution(training_dataframe=training_dataframe, model_file=model_file)

In [ ]:
training.plot_model_feature_importance(training_dataframe=training_dataframe, model_file=model_file)

### Validate 

In [ ]:
if not test_prediction_file.exists():

    predictions, satellite_data = training.predict_site(test_satellite_file=test_satellite_file, polygon_file=test_polygon_file, model_file=model_file)
    utils.save_netcdf(predictions, test_prediction_file)
else:
    print(f"Prediction file '{test_prediction_file.name}' already eixsts. Delete if you want to repredict.")
    satellite_data = utils.load_satellite(filename=test_satellite_file)
    predictions = utils.load_classification(filename=test_prediction_file, chunks=None)

print(
    f"Satellite training classes present: {[key for key, value in satellite_classes.items() if value in training_dataframe['satellite_class_id'].unique()]}. "
    f"Predicted classes present: {[key for key, value in satellite_classes.items() if value in numpy.unique(predictions.data)]}"
     )

In [ ]:
training.confusion_matrix_of_site(
    test_uav_file=test_uav_file,
    uav_labels_file=uav_labels_file,
    prediction_file=test_prediction_file,
    satellite_classes=satellite_classes,
    satellite_from_uav_classes=satellite_from_uav_classes,
    uav_classes_to_ignore=uav_classes_to_ignore,
    polygon_file=test_polygon_file)

## Unfinished experimental code

In [ ]:
training.confusion_matrix_of_site_satellite_resolution(
    test_uav_file=test_uav_file,
    uav_labels_file=uav_labels_file,
    prediction_file=test_prediction_file,
    satellite_classes=satellite_classes,
    satellite_from_uav_classes=satellite_from_uav_classes,
    uav_classes_to_ignore=uav_classes_to_ignore,
    polygon_file=test_polygon_file)

In [ ]:
import numpy
import xarray
import scipy

# Create the DataArray
da = xarray.DataArray(
    data=[[1,1,1,1],[1,2,2,2],[2,3,4,4],[3,3,4,5]],
    dims=("x", "y"),
    coords={"x": [10, 20, 30, 40], "y": [1, 2, 3, 4]},
    name="test_data"
)

def get_mode(arr, axis):
    # returns the mode values only; ignore counts
    return scipy.stats.mode(arr, axis=axis).mode

da.coarsen(x=2, y=2).reduce(get_mode)